# Real-Time Indian Stock Price Prediction and Signal Dashboard
### End-to-End Quantitative Machine Learning Exploration (NSE/BSE Data)

This notebook demonstrates the end-to-end data pipeline, feature engineering, zero-leakage preprocessing, model training (XGBoost & LSTM), walk-forward backtesting, and rule-based signal generation for Indian equities.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.download_data import download_and_save_stock_data
from src.features.technical_indicators import calculate_technical_indicators, create_prediction_targets
from src.features.preprocessor import preprocess_stock_data
from src.models.xgboost_model import StockXGBoostModel
from src.models.lstm_model import StockLSTMModel
from src.evaluation.metrics import calculate_metrics, compare_models
from src.signals.signal_engine import SignalEngine
from config.stocks import NIFTY_50_UNIVERSE

## 1. Fetch & Validate Historical Data from NSE

In [ ]:
symbol = 'RELIANCE.NS'
df_raw, report = download_and_save_stock_data(symbol=symbol, period='2y', interval='1d')
print('Data Validation Report:', report)
df_raw.tail()

## 2. Technical Indicators & Feature Engineering

In [ ]:
df_features = calculate_technical_indicators(df_raw)
df_targets = create_prediction_targets(df_features)
print(f'Engineered {len(df_features.columns)} features.')
df_targets[['Close', 'SMA_20', 'EMA_20', 'RSI_14', 'MACD', 'Target_Next_Close', 'Target_Direction']].tail()

## 3. Strict Chronological Preprocessing & Anti-Leakage Scaling

In [ ]:
dataset = preprocess_stock_data(df_raw)
print(f'Train size: {len(dataset.X_train)} | Val size: {len(dataset.X_val)} | Test size: {len(dataset.X_test)}')
print(f'LSTM sequence shape (Samples, Lookback, Features): {dataset.X_train_seq.shape}')

## 4. Train & Evaluate Models

In [ ]:
# XGBoost
xgb = StockXGBoostModel()
xgb.train(dataset, symbol=symbol)
xgb_preds = xgb.predict(dataset.X_test)

# Evaluation
metrics = calculate_metrics(
    y_true=dataset.y_test.values,
    y_pred=xgb_preds,
    current_prices=dataset.X_test['Close'].values,
    model_name='XGBoost'
)
pd.DataFrame([metrics])

## 5. Generate Multi-Factor Consensus Research Signal

In [ ]:
engine = SignalEngine()
latest_price = float(df_raw['Close'].iloc[-1])
next_day_pred = float(xgb_preds[-1])

signal = engine.generate_signal(
    symbol=symbol,
    current_price=latest_price,
    xgb_prediction=next_day_pred,
    latest_features=df_features.iloc[-1]
)

print(f'Signal: {signal.signal}')
print(f'Predicted Return: {signal.predicted_return_pct}%')
print('Reasons:')
for r in signal.reasons:
    print(f'  - {r}')